In [92]:
import pandas as pd
import numpy as np

leagues = ["E0", "D1", "SP1", "I1"]
seasons = ["2122", "2223", "2324", "2425"]

df_list = []
for l in leagues:
    for s in seasons:
        url_csv = f"https://www.football-data.co.uk/mmz4281/{s}/{l}.csv"
        df_part = pd.read_csv(url_csv)
        df_part["season"] = s
        df_part["league"] = l
        df_list.append(df_part)

df = pd.concat(df_list)
print(df.shape)

strats = df.copy()

(5784, 134)


In [93]:
# S1 - Alacsony odds: bet

odds_min, odds_max = 1.3, 1.6

def profit_s1_all(home_odds, away_odds, result):
    if (home_odds > odds_min) & (home_odds < odds_max):
        profit = home_odds - 1 if result == "H" else -1
    elif (away_odds > odds_min) & (away_odds < odds_max):
        profit = away_odds - 1 if result == "A" else -1
    else:
        profit = 0
    
    return profit

## vari1: csak home
strats["s1v1_profit"] = strats.apply(lambda x: profit_s1_all(x["B365H"], 0, x["FTR"]), axis=1)

## vari2: mindenki
strats["s1v2_profit"] = strats.apply(lambda x: profit_s1_all(x["B365H"], x["B365A"], x["FTR"]), axis=1)

## vari3: csak away
strats["s1v3_profit"] = strats.apply(lambda x: profit_s1_all(0, x["B365A"], x["FTR"]), axis=1)

for col in ["s1v1_profit", "s1v2_profit", "s1v3_profit"]:
    bets = strats[strats[col] != 0]
    print(col,
          "bets:", len(bets),
          "ROI:", bets[col].sum() / len(bets),
          "Hit rate:", (bets[col] > 0).mean()
         )


s1v1_profit bets: 765 ROI: -0.011529411764705885 Hit rate: 0.6888888888888889
s1v2_profit bets: 1072 ROI: -0.011026119402985068 Hit rate: 0.6856343283582089
s1v3_profit bets: 307 ROI: -0.00977198697068403 Hit rate: 0.6775244299674267


In [94]:
# Túlértékelt nagy favoritok

# S2 – Túlértékelt nagy favoritok fade-elése

fav_max_odds = 1.40


def profit_s2_fade(home_odds, away_odds, draw_odds, result, mode="all"):
    profit = 0

    # Away csapat túl nagy favorit -> HOME vagy DRAW
    if (mode in ["away", "all"]) and (away_odds > 0) and (away_odds < fav_max_odds):
        if result in ["H", "D"]:
            profit = (1 / (1/home_odds + 1/draw_odds)) - 1
        else:
            profit = -1

    # Home csapat túl nagy favorit -> AWAY vagy DRAW
    elif (mode in ["home", "all"]) and (home_odds > 0) and (home_odds < fav_max_odds):
        if result in ["A", "D"]:
            profit = (1 / (1/away_odds + 1/draw_odds)) - 1
        else:
            profit = -1

    return profit

strats["s2v1_profit"] = strats.apply(
    lambda x: profit_s2_fade(x["AvgH"], x["AvgA"], x["AvgD"], x["FTR"], mode="away"),
    axis=1
)

strats["s2v2_profit"] = strats.apply(
    lambda x: profit_s2_fade(x["AvgH"], x["AvgA"], x["AvgD"], x["FTR"], mode="home"),
    axis=1
)


strats["s2v3_profit"] = strats.apply(
    lambda x: profit_s2_fade(x["AvgH"], x["AvgA"], x["AvgD"], x["FTR"], mode="all"),
    axis=1
)


for col in ["s2v1_profit", "s2v2_profit", "s2v3_profit"]:
    bets = strats[strats[col] != 0]
    print(col,
          "bets:", len(bets),
          "ROI:", bets[col].sum() / len(bets),
          "Hit rate:", (bets[col] > 0).mean()
         )


s2v1_profit bets: 143 ROI: -0.2286417967300358 Hit rate: 0.2097902097902098
s2v2_profit bets: 670 ROI: -0.12125068800924839 Hit rate: 0.22238805970149253
s2v3_profit bets: 813 ROI: -0.14013989901425775 Hit rate: 0.2201722017220172


In [95]:
# S3 - Draw value

def profit_s3_draw_value(home_odds, away_odds, draw_odds, result):
    if (
        abs(home_odds - away_odds) <= 0.30
        and 2.20 <= home_odds <= 2.90
        and 2.20 <= away_odds <= 2.90
        and draw_odds >= 3.40
    ):
        return draw_odds - 1 if result == "D" else -1
    return 0

strats["s3v1_profit"] = strats.apply(
    lambda x: profit_s3_draw_value(x["AvgH"], x["AvgA"], x["AvgD"], x["FTR"]),
    axis=1
)

bets = strats[strats["s3v1_profit"] != 0]
print(
    "bets:", len(bets),
    "ROI:", bets["s3v1_profit"].sum() / len(bets),
    "Hit rate:", (bets["s3v1_profit"] > 0).mean()
)


bets: 197 ROI: -0.09517766497461928 Hit rate: 0.25380710659898476


In [98]:
# S1V3 vizsgálata

def profit_away_fav(home_odds, away_odds, result, lo, hi):
    if lo <= away_odds <= hi:
        return away_odds - 1 if result == "A" else -1
    return 0

import numpy as np
import pandas as pd

results = []

odds_lows  = np.arange(1.20, 1.80, 0.05)
odds_highs = np.arange(1.30, 2.00, 0.05)

for lo in odds_lows:
    for hi in odds_highs:
        if hi <= lo:
            continue

        profits = strats.apply(
            lambda x: profit_away_fav(x["AvgH"], x["AvgA"], x["FTR"], lo, hi),
            axis=1
        )

        bets = profits[profits != 0]
        if len(bets) < 100:
            continue

        results.append({
            "away_odds_lo": round(lo, 2),
            "away_odds_hi": round(hi, 2),
            "bets": len(bets),
            "ROI": bets.sum() / len(bets),
            "hit_rate": (bets > 0).mean()
        })

grid_df = pd.DataFrame(results).sort_values("ROI", ascending=False)
display(grid_df.head(10))

# Grid alapján
ODDS_LO, ODDS_HI = 1.75, 1.95

league_results = []

for league in strats["league"].unique():
    df = strats[strats["league"] == league]
    profits = df.apply(
        lambda x: profit_away_fav(x["AvgH"], x["AvgA"], x["FTR"], ODDS_LO, ODDS_HI),
        axis=1
    )

    bets = profits[profits != 0]

    league_results.append({
        "league": league,
        "bets": len(bets),
        "ROI": bets.sum() / len(bets),
        "hit_rate": (bets > 0).mean()
    })

league_df = (
    pd.DataFrame(league_results)
    .sort_values("ROI", ascending=False)
)

display(league_df)


,away_odds_lo,away_odds_hi,bets,ROI,hit_rate
98,1.80,1.90,151,0.068411,0.576159
99,1.80,1.95,220,0.058136,0.563636
96,1.75,1.90,226,0.044513,0.570796
97,1.75,1.95,295,0.042441,0.562712
95,1.75,1.85,144,0.038750,0.576389
46,1.40,1.50,111,0.037748,0.711712
25,1.30,1.50,206,0.036505,0.737864
2,1.20,1.50,245,0.035265,0.751020
90,1.65,1.95,457,0.035011,0.575492
89,1.65,1.90,388,0.034897,0.582474


,league,bets,ROI,hit_rate
2,SP1,68,0.125294,0.602941
3,I1,108,0.074444,0.583333
0,E0,86,-0.014186,0.534884
1,D1,50,-0.116400,0.480000


In [99]:
# S1V3 - odds buckets

strats["away_bucket"] = pd.cut(
    strats["AvgA"],
    bins=np.arange(1.20, 3.00, 0.10)
)

bucket_results = []

for bucket, df in strats.groupby("away_bucket"):
    profits = df.apply(
        lambda x: profit_away_fav(x["AvgH"], x["AvgA"], x["FTR"], 0, 99),
        axis=1
    )

    bets = profits[profits != 0]
    if len(bets) < 50:
        continue

    bucket_results.append({
        "away_bucket": str(bucket),
        "bets": len(bets),
        "ROI": bets.sum() / len(bets),
        "hit_rate": (bets > 0).mean(),
        "avg_odds": df["AvgA"].mean()
    })

bucket_df = (
    pd.DataFrame(bucket_results)
    .sort_values("ROI", ascending=False)
)

bucket_df


C:\Users\Adam\AppData\Local\Temp\ipykernel_17596\2009848110.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for bucket, df in strats.groupby("away_bucket"):


,away_bucket,bets,ROI,hit_rate,avg_odds
8,"(2.1, 2.2]",171,0.071988,0.497076,2.155146
5,"(1.8, 1.9]",151,0.068411,0.576159,1.855033
10,"(2.3, 2.4]",149,0.058523,0.449664,2.355369
9,"(2.2, 2.3]",162,0.043519,0.462963,2.252963
1,"(1.4, 1.5]",111,0.037748,0.711712,1.456937
7,"(2.0, 2.1]",130,0.029385,0.500000,2.058615
0,"(1.3, 1.4]",91,0.023407,0.758242,1.351099
6,"(1.9, 2.0]",128,0.021953,0.523438,1.952734
15,"(2.8, 2.9]",158,0.014241,0.354430,2.858228
13,"(2.6, 2.7]",142,0.010141,0.380282,2.655493
